# VA-AFS Colab Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kiim-Miin-Su/VA-AFS/blob/main/colab_run.ipynb)

웹 Colab에서는 `Runtime -> Change runtime type -> GPU`를 먼저 선택한다. 기본 설정은 빠른 smoke test만 실행한다. 발표용 80 epoch 실험은 아래 설정 셀에서 `RUN_PRESENTATION = True`로 바꾼 뒤 실행한다.


In [ ]:
# @title 1. Mount Google Drive
from pathlib import Path
import os
import shutil
import subprocess
import sys

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    print('Local kernel: skip Google Drive mount.')


## 데이터 위치

Google Drive 공유 폴더의 zip 파일 4개를 본인 Drive의 아래 경로에 둔다.

```text
/content/drive/MyDrive/AFS/data/
  all_sqe.zip
  nturgbd_skeletons_s001_to_s017.zip
  nturgbd_skeletons_s018_to_s032.zip
  videos.zip
```


In [ ]:
# @title 2. Runtime config
REPO_URL = 'https://github.com/Kiim-Miin-Su/VA-AFS.git'
BRANCH = 'main'
DATA_DIR = '/content/drive/MyDrive/AFS/data' if IN_COLAB else str((Path.cwd() / 'data').resolve())
PROJECT_ROOT = Path('/content/src') if IN_COLAB else Path.cwd()
FORCE_RECLONE = False

RUN_SMOKE_TEST = True
RUN_PRESENTATION = True
RUN_FULL_DATA = False
SHOW_FIGURES = RUN_PRESENTATION
BACKUP_RESULTS_TO_DRIVE = True

SMOKE_SAMPLE_SIZE = 200
SMOKE_NUM_EPOCH = 2
SMOKE_BATCH_SIZE = 8

PRESENTATION_SAMPLE_SIZE = 16000
PRESENTATION_NUM_EPOCH = 80
PRESENTATION_BATCH_SIZE = 64
PRESENTATION_TEST_BATCH_SIZE = 64
PRESENTATION_NUM_WORKER = 2

print('IN_COLAB:', IN_COLAB)
print('DATA_DIR:', DATA_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)


In [ ]:
# @title 3. Prepare repository

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'VA-AFS').exists() and (candidate / 'BlockGCN').exists():
            return candidate
    return start

if IN_COLAB:
    if FORCE_RECLONE and PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    if not (PROJECT_ROOT / 'VA-AFS').exists():
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        print(f'skip clone: {PROJECT_ROOT} already exists')
else:
    PROJECT_ROOT = find_project_root(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('cwd:', Path.cwd())
print('setup_colab.py exists:', Path('setup_colab.py').exists())


In [ ]:
# @title 4. Check data zip files
required_zips = [
    'all_sqe.zip',
    'nturgbd_skeletons_s001_to_s017.zip',
    'nturgbd_skeletons_s018_to_s032.zip',
    'videos.zip',
]

data_dir = Path(DATA_DIR)
missing = [name for name in required_zips if not (data_dir / name).exists()]
print('DATA_DIR:', data_dir)
if data_dir.exists():
    for path in sorted(data_dir.iterdir()):
        if path.name in required_zips:
            print(f'{path.name}: {path.stat().st_size / (1024 ** 2):.1f} MB')
if missing:
    raise FileNotFoundError('Missing data zip files: ' + ', '.join(missing))


In [ ]:
# @title 5. Extract data and install Colab requirements
if IN_COLAB:
    subprocess.run(
        [sys.executable, 'setup_colab.py', '--data_dir', DATA_DIR, '--install', '--verify'],
        check=True,
    )
else:
    print('Local kernel: skip Colab setup. Use local requirements/data paths.')


In [ ]:
# @title 6. Check accelerator
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    DEVICE_ARG = '0'
    print('cuda device:', torch.cuda.get_device_name(0))
else:
    DEVICE_ARG = None
    print('No CUDA GPU detected. Colab menu에서 GPU 런타임을 선택해야 학습이 빨라진다.')


## Smoke test

런타임, 데이터, 의존성 확인용 짧은 실행이다. `RUN_SMOKE_TEST = True`이면 `Run all`에서 자동 실행된다.


In [ ]:
# @title 7. Run smoke test
if RUN_SMOKE_TEST:
    cmd = [
        sys.executable,
        'VA-AFS/run_colab_pipeline.py',
        '--sample_size', str(SMOKE_SAMPLE_SIZE),
        '--num_epoch', str(SMOKE_NUM_EPOCH),
        '--batch_size', str(SMOKE_BATCH_SIZE),
        '--test_batch_size', str(SMOKE_BATCH_SIZE),
        '--num_worker', '1',
    ]
    if DEVICE_ARG is not None:
        cmd.extend(['--device', DEVICE_ARG])
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('skip smoke test')


## Presentation run

발표용 실행은 오래 걸린다. 설정 셀에서 `RUN_PRESENTATION = True`로 바꾸면 실행된다. 기본값은 `sample_size=3000`, `epoch=80`이다.


In [ ]:
# @title 8. Run presentation pipeline
if RUN_PRESENTATION:
    cmd = [
        sys.executable,
        'VA-AFS/run_colab_pipeline.py',
        '--sample_size', str(PRESENTATION_SAMPLE_SIZE),
        '--num_epoch', str(PRESENTATION_NUM_EPOCH),
        '--batch_size', str(PRESENTATION_BATCH_SIZE),
        '--test_batch_size', str(PRESENTATION_TEST_BATCH_SIZE),
        '--num_worker', str(PRESENTATION_NUM_WORKER),
    ]
    if DEVICE_ARG is not None:
        cmd.extend(['--device', DEVICE_ARG])
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('skip presentation run. Set RUN_PRESENTATION = True in the config cell to run it.')


In [ ]:
# @title 9. Optional full NTU60 run
if RUN_FULL_DATA:
    cmd = [
        sys.executable,
        'VA-AFS/run_colab_pipeline.py',
        '--full_data',
        '--num_epoch', str(PRESENTATION_NUM_EPOCH),
        '--batch_size', str(PRESENTATION_BATCH_SIZE),
        '--test_batch_size', str(PRESENTATION_TEST_BATCH_SIZE),
        '--num_worker', str(PRESENTATION_NUM_WORKER),
    ]
    if DEVICE_ARG is not None:
        cmd.extend(['--device', DEVICE_ARG])
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    print('skip full-data run')


## Result visualization


In [ ]:
# @title 10. Load visualization helpers
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    def display(value):
        print(value)

def _resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'VA-AFS').exists() and (candidate / 'BlockGCN').exists():
            return candidate
    colab_root = Path('/content/src')
    if colab_root.exists():
        return colab_root
    return cwd


ROOT = _resolve_project_root()
OUTPUT_DIR = ROOT / 'VA-AFS' / 'outputs'
PLOT_DIR = OUTPUT_DIR / 'presentation_plots'
print('Project root:', ROOT)
print('Presentation plot dir:', PLOT_DIR)


def _parse_run_name(path):
    name = Path(path).stem
    sample_match = re.search(r'subset_?(?P<sample>\d+)', name)
    epoch_match = re.search(r'_e(?P<epoch>\d+)$', name)
    tau_match = re.search(r'tau(?P<tau>[0-9.]+)', name)
    k_match = re.search(r'_k(?P<k>\d+)', name)
    w_match = re.search(r'_w(?P<w>\d+)', name)
    variant = 'vaafs' if 'vaafs' in name.lower() else 'original'
    is_full_data = 'ntu_full' in name.lower()

    return {
        'run': name,
        'sample_size': int(sample_match.group('sample')) if sample_match else None,
        'num_epoch': int(epoch_match.group('epoch')) if epoch_match else None,
        'variant': variant,
        'full_data': is_full_data,
        'tau': float(tau_match.group('tau')) if tau_match else None,
        'k_max': int(k_match.group('k')) if k_match else None,
        'window_size': int(w_match.group('w')) if w_match else None,
    }


def _normalize_accuracy(value):
    value = float(value)
    return value / 100.0 if value > 1.0 else value


def _read_accuracy(log_path):
    text = Path(log_path).read_text(errors='ignore')
    patterns = [
        r'Parsed Top-1 accuracy:\s*([0-9.]+)',
        r'Accuracy:\s*([0-9.]+)',
        r'Top1:\s*([0-9.]+)%',
        r'Top1 Acc:\s*([0-9.]+)%',
    ]
    matches = []
    for pattern in patterns:
        matches.extend(re.findall(pattern, text))
    if not matches:
        return None
    return _normalize_accuracy(matches[-1])


def collect_accuracy_results(acc_dir=None):
    acc_dir = Path(acc_dir) if acc_dir is not None else OUTPUT_DIR / 'blockgcn_acc'
    rows = []
    for log_path in sorted(Path(acc_dir).glob('**/log.txt')):
        accuracy = _read_accuracy(log_path)
        if accuracy is None:
            continue
        info = _parse_run_name(log_path.parent)
        rows.append({
            **info,
            'accuracy': accuracy,
            'accuracy_percent': accuracy * 100.0,
            'log_path': str(log_path),
        })
    return pd.DataFrame(rows)


def _valid_frame_counts(values):
    if values.ndim < 2:
        raise ValueError(f'Expected sequence array with at least 2 dimensions, got {values.shape}')
    feature_axes = tuple(range(2, values.ndim))
    return np.any(values != 0, axis=feature_axes).sum(axis=1).astype(np.int32)


def _candidate_original_npz_paths(info):
    if info.get('full_data'):
        return [ROOT / 'BlockGCN' / 'data' / 'ntu' / 'NTU60_CS.npz']
    sample_size = info.get('sample_size')
    if sample_size is None:
        return []
    return [ROOT / 'BlockGCN' / 'data' / f'ntu_subset_{sample_size}' / 'NTU60_CS.npz']


def _load_frame_counts(npz_path, split, info=None):
    npz_path = Path(npz_path)
    info = dict(info) if info is not None else _parse_run_name(npz_path)
    original_key = f'{split}_original_counts'
    selected_key = f'{split}_selected_counts'
    x_key = f'x_{split}'

    with np.load(npz_path) as data:
        if original_key in data and selected_key in data:
            return data[original_key], data[selected_key], 'metadata'
        if x_key not in data:
            return None, None, 'missing'
        selected_counts = _valid_frame_counts(data[x_key])

    for original_npz in _candidate_original_npz_paths(info):
        if not original_npz.exists():
            continue
        with np.load(original_npz) as original_data:
            if x_key not in original_data:
                continue
            original_counts = _valid_frame_counts(original_data[x_key])
        if len(original_counts) == len(selected_counts):
            return original_counts, selected_counts, 'inferred'

    return None, None, 'missing'


def collect_frame_ratio_results(npz_dir=None):
    npz_dir = Path(npz_dir) if npz_dir is not None else OUTPUT_DIR / 'blockgcn_npz'
    rows = []
    for npz_path in sorted(Path(npz_dir).glob('*.npz')):
        info = _parse_run_name(npz_path)
        for split in ('train', 'test'):
            original_counts, selected_counts, counts_source = _load_frame_counts(npz_path, split, info)
            if original_counts is None or selected_counts is None:
                continue

            valid = original_counts > 0
            total_original = int(original_counts[valid].sum())
            total_selected = int(selected_counts[valid].sum())
            processed_ratio = total_selected / max(total_original, 1)

            rows.append({
                **info,
                'split': split,
                'samples': int(valid.sum()),
                'original_frames': total_original,
                'selected_frames': total_selected,
                'processed_frame_ratio': processed_ratio,
                'frame_reduction_ratio': 1.0 - processed_ratio,
                'counts_source': counts_source,
                'npz_path': str(npz_path),
            })
    return pd.DataFrame(rows)


def _filter_latest(df, sample_size=None, num_epoch=None, split=None, variant=None, full_data=None):
    if df.empty:
        return df
    result = df.copy()
    if sample_size is not None and 'sample_size' in result:
        result = result[result['sample_size'] == sample_size]
    if num_epoch is not None and 'num_epoch' in result:
        result = result[result['num_epoch'] == num_epoch]
    if split is not None and 'split' in result:
        result = result[result['split'] == split]
    if variant is not None and 'variant' in result:
        result = result[result['variant'] == variant]
    if full_data is not None and 'full_data' in result:
        result = result[result['full_data'] == full_data]
    return result


def _save_and_show(fig, save_path):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200, bbox_inches='tight')
    print(f'Saved plot: {save_path}')
    plt.show()
    if Image is not None:
        display(Image(filename=str(save_path)))


def plot_frame_ratio(sample_size=None, split='test', full_data=False, save_path=None):
    if sample_size is None:
        sample_size = PRESENTATION_SAMPLE_SIZE
    summary = _filter_latest(
        collect_frame_ratio_results(),
        sample_size=None if full_data else sample_size,
        split=split,
        variant='vaafs',
        full_data=full_data,
    )
    if summary.empty:
        npz_dir = OUTPUT_DIR / 'blockgcn_npz'
        available = ', '.join(p.name for p in sorted(npz_dir.glob('*.npz'))) or 'none'
        raise FileNotFoundError(
            f'No VA-AFS frame ratio result found for sample={"full" if full_data else sample_size}, '
            f'split={split}. Searched: {npz_dir}. Available npz files: {available}. '
            'Run the presentation pipeline cell, or rerun it with --force_vaafs if the npz was created by an older notebook.'
        )

    row = summary.sort_values('npz_path').iloc[-1]
    original_counts, selected_counts, _ = _load_frame_counts(row['npz_path'], split, row)
    if original_counts is None or selected_counts is None:
        raise FileNotFoundError(f'Could not load frame counts from {row["npz_path"]}')

    valid = original_counts > 0
    ratios = selected_counts[valid] / original_counts[valid]
    processed_ratio = float(row['processed_frame_ratio'])
    reduction_ratio = float(row['frame_reduction_ratio'])

    fig, (ax_bar, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.8))
    bars = ax_bar.bar(
        ['Processed', 'Skipped'],
        [processed_ratio, reduction_ratio],
        color=['#2E7D32', '#C62828'],
    )
    ax_bar.set_ylim(0, 1)
    ax_bar.set_ylabel('Frame ratio')
    ax_bar.set_title('Total Frame Ratio')
    ax_bar.grid(axis='y', alpha=0.25)
    for bar in bars:
        value = bar.get_height()
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.025,
            f'{value * 100:.1f}%',
            ha='center',
            va='bottom',
            fontsize=11,
            fontweight='bold',
        )

    ax_hist.hist(ratios, bins=np.linspace(0, 1, 11), color='#1565C0', edgecolor='white')
    ax_hist.axvline(processed_ratio, color='#C62828', linestyle='--', label=f'Mean {processed_ratio * 100:.1f}%')
    ax_hist.set_xlim(0, 1)
    ax_hist.set_xlabel('Processed ratio per sample')
    ax_hist.set_ylabel('Sample count')
    ax_hist.set_title('Per-sample Ratio Distribution')
    ax_hist.legend()
    ax_hist.grid(axis='y', alpha=0.25)

    fig.suptitle(
        f'VA-AFS Frame Selection | sample={"full" if full_data else sample_size}, split={split}, '
        f'tau={row["tau"]}, k={row["k_max"]}, window={row["window_size"]}',
        fontsize=13,
        fontweight='bold',
    )
    fig.tight_layout()
    if save_path is None:
        run_label = 'full' if full_data else f'subset{sample_size}'
        save_path = PLOT_DIR / f'{run_label}_{split}_frame_ratio.png'
    _save_and_show(fig, save_path)
    return summary


def plot_accuracy(sample_size=None, num_epoch=None, full_data=False, save_path=None):
    if sample_size is None:
        sample_size = PRESENTATION_SAMPLE_SIZE
    if num_epoch is None:
        num_epoch = PRESENTATION_NUM_EPOCH
    df = _filter_latest(
        collect_accuracy_results(),
        sample_size=None if full_data else sample_size,
        num_epoch=num_epoch,
        full_data=full_data,
    )
    if df.empty:
        acc_dir = OUTPUT_DIR / 'blockgcn_acc'
        available_logs = sorted(acc_dir.glob('**/log.txt'))
        available_text = '\n'.join(f'  - {p.relative_to(ROOT)}' for p in available_logs[:20]) or '  - none'
        if len(available_logs) > 20:
            available_text += f'\n  ... {len(available_logs) - 20} more'
        target = 'full data' if full_data else f'sample_size={sample_size}'
        raise FileNotFoundError(
            f'Accuracy logs were not found for {target}, epoch={num_epoch}.\n'
            f'Searched: {acc_dir}\n'
            f'Available log.txt files:\n{available_text}\n'
            'In Colab, rerun the presentation pipeline cell with the same sample_size and num_epoch. '
            'Existing subset, preprocessing, training checkpoint, and VA-AFS npz files are reused; missing eval logs are regenerated.'
        )

    order = ['original', 'vaafs']
    df = df.sort_values('variant').drop_duplicates('variant', keep='last')
    df['variant'] = pd.Categorical(df['variant'], categories=order, ordered=True)
    df = df.sort_values('variant')

    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    colors = ['#455A64' if variant == 'original' else '#2E7D32' for variant in df['variant'].astype(str)]
    bars = ax.bar(df['variant'].astype(str), df['accuracy_percent'], color=colors)
    ax.set_ylim(0, max(100, float(df['accuracy_percent'].max()) + 5))
    ax.set_ylabel('Top-1 accuracy (%)')
    ax.set_title(f'BlockGCN Accuracy Comparison | sample={"full" if full_data else sample_size}, epoch={num_epoch}')
    ax.grid(axis='y', alpha=0.25)
    for bar in bars:
        value = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.8,
            f'{value:.2f}%',
            ha='center',
            va='bottom',
            fontsize=11,
            fontweight='bold',
        )

    if {'original', 'vaafs'}.issubset(set(df['variant'].astype(str))):
        original = float(df.loc[df['variant'].astype(str) == 'original', 'accuracy_percent'].iloc[-1])
        vaafs = float(df.loc[df['variant'].astype(str) == 'vaafs', 'accuracy_percent'].iloc[-1])
        ax.text(
            0.5,
            0.95,
            f'Delta: {vaafs - original:+.2f} pp',
            transform=ax.transAxes,
            ha='center',
            va='top',
            bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': '#B0BEC5'},
        )

    fig.tight_layout()
    if save_path is None:
        run_label = 'full' if full_data else f'subset{sample_size}'
        save_path = PLOT_DIR / f'{run_label}_e{num_epoch}_accuracy.png'
    _save_and_show(fig, save_path)
    return df


def show_threshold_selection_plots(limit=3, plot_dir=None):
    plot_dir = Path(plot_dir) if plot_dir is not None else OUTPUT_DIR / 'threshold' / 'plots'
    plots = sorted(Path(plot_dir).glob('*_selection.png'))[-limit:]
    if not plots:
        raise FileNotFoundError('Saved threshold selection plots were not found.')
    if Image is None:
        return plots
    for plot_path in plots:
        print(plot_path)
        display(Image(filename=str(plot_path)))
    return plots


def show_presentation_figures(sample_size=None, num_epoch=None, split='test', full_data=False):
    if sample_size is None:
        sample_size = PRESENTATION_SAMPLE_SIZE
    if num_epoch is None:
        num_epoch = PRESENTATION_NUM_EPOCH
    frame_summary = plot_frame_ratio(sample_size=sample_size, split=split, full_data=full_data)
    acc_summary = plot_accuracy(sample_size=sample_size, num_epoch=num_epoch, full_data=full_data)
    display(frame_summary)
    display(acc_summary)
    return frame_summary, acc_summary


In [ ]:
# @title 11. Show presentation figures
if SHOW_FIGURES:
    frame_summary, acc_summary = show_presentation_figures(split='test')
else:
    print('skip figures. Set SHOW_FIGURES = True after a presentation run.')


In [ ]:
# @title 12. Optional Drive backup
if BACKUP_RESULTS_TO_DRIVE:
    from datetime import datetime

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    backup_base = Path('/content/drive/MyDrive/AFS') if IN_COLAB else PROJECT_ROOT
    backup_base.mkdir(parents=True, exist_ok=True)
    archive_base = backup_base / f'va_afs_outputs_{stamp}'
    shutil.make_archive(str(archive_base), 'zip', root_dir=PROJECT_ROOT / 'VA-AFS' / 'outputs')
    print(f'backup saved: {archive_base}.zip')
else:
    print('skip Drive backup')
